# Main-text figures (Fig. 1-7)

A walk-through of how the article's figures are produced.  Every figure is built in
two steps, and this notebook shows both:

| Step | Reads | Writes |
|------|-------|--------|
| **build** | `results/metrics/**.xlsx` | `results/figure_data/figN_*.csv` |
| **draw**  | `results/figure_data/figN_*.csv` | `figures/main/figN_*.{png,pdf}` |

Because each plot is drawn from a small committed CSV, restyling a figure never
re-reads a metric table and never recomputes anything.

The same thing runs from the command line in one line, which is what the pipeline
itself uses:

```bash
python run_pipeline.py figures --which main
```

In [1]:
# Run from anywhere: put the repository root on the import path.
import sys, pathlib

_root = next(p for p in pathlib.Path.cwd().resolve().parents
             if (p / 'glmbench').is_dir())
sys.path.insert(0, str(_root))


In [2]:
from glmbench import config, models
from glmbench.figures import main, style

style.set_style()
chosen = models.resolve(models.PUBLISHED)

print("repository :", config.ROOT)
print("metrics    :", config.relative(config.METRICS))
print("numbers    :", config.relative(config.FIGURE_DATA))
print("figures    :", config.relative(config.FIG_MAIN))
print("models     :", ", ".join(model.display for model in chosen))

repository : C:\Users\aleks\Documents\glm_vectorizing
metrics    : results/metrics
numbers    : results/figure_data
figures    : figures/main
models     : k-mer, DNABERT-2, DNABERT-S, ViraLM, Evo2-7B


## Which layer each figure reports

Fig. 1 and 2 *are* the reference benchmark, so they show its own best layer.

Fig. 4, 5 and 6 report every model at the layer that benchmark chose -- never at the
layer that happens to score best on the unseen order itself.  One layer per model is
used across all of them, so nothing is tuned on the numbers being reported.  The
choice is recorded in `results/metrics/selection.json`:

In [3]:
from glmbench import selection

selection.load()

{'dnabert2': {'layer': 3,
  'k': None,
  'score': 0.555197,
  'chosen_by': 'reference P5'},
 'dnabertS': {'layer': 4,
  'k': None,
  'score': 0.603262,
  'chosen_by': 'reference P5'},
 'viralm': {'layer': 2,
  'k': None,
  'score': 0.637392,
  'chosen_by': 'reference P5'},
 'evo2': {'layer': 26,
  'k': None,
  'score': 0.612645,
  'chosen_by': 'reference P5'}}

## Build: metric tables -> the numbers behind each figure

In [4]:
main.build_fig1(chosen)      # reference benchmark: ARI, P@5, silhouette
main.build_fig2(chosen)      # the same, layer by layer
main.build_fig3()            # k-mer word-length sweep
main.build_fig4(chosen)      # unseen orders, forward and reverse complement
main.build_fig5(chosen)      # forward vs reverse-complement proximity
main.build_fig6(chosen)      # P@k against k
main.build_fig7(chosen)      # compactness against separation, per layer

  numbers -> results/figure_data/fig1_reference_benchmark.csv
  numbers -> results/figure_data/fig2_layerwise.csv
  numbers -> results/figure_data/fig3_kmer_sweep.csv


  numbers -> results/figure_data/fig4_newvirus_p5.csv


  numbers -> results/figure_data/fig5_cross_p5.csv


  numbers -> results/figure_data/fig6_pk_vs_k.csv
  numbers -> results/figure_data/fig7_pk_vs_cross.csv


,model,layer,p_at_10,cross_p_at_10
0,kmer,-,0.452475,0.115544
1,dnabert2,0,0.326914,0.176089
2,dnabert2,1,0.402281,0.186492
3,dnabert2,2,0.448289,0.192885
4,dnabert2,3,0.447687,0.225849
...,...,...,...,...
66,evo2,27,0.367583,0.253180
67,evo2,28,0.142137,0.252243
68,evo2,29,0.105106,0.244439
69,evo2,30,0.064629,0.273423


Each call returns the data frame it wrote, so a figure's numbers can be inspected
before anything is drawn:

In [5]:
main.build_fig4(chosen).pivot(index="model", columns="group", values="mean").round(4)

  numbers -> results/figure_data/fig4_newvirus_p5.csv


group,Egovirales,Egovirales (RC),Mriyavirales,Mriyavirales (RC)
model,,,,
dnabert2,0.7806,0.7657,0.4693,0.4381
dnabertS,0.8510,0.8362,0.5652,0.5513
evo2,0.7072,0.6920,0.5560,0.5381
kmer,0.8145,0.8120,0.3981,0.3790
viralm,0.8808,0.8711,0.4701,0.4532


## Draw: the numbers -> the figures

This step reads only `results/figure_data/`; `data/` and `results/metrics/` are not
touched.

In [6]:
main.draw_all(chosen)

  figure  -> figures/main/fig1_reference_benchmark.{png,pdf}


  figure  -> figures/main/fig2_layerwise.{png,pdf}


  figure  -> figures/main/fig3_kmer_sweep.{png,pdf}


  figure  -> figures/main/fig4_newvirus_p5.{png,pdf}


  figure  -> figures/main/fig5_cross_p5.{png,pdf}


  figure  -> figures/main/fig6_pk_vs_k.{png,pdf}


  figure  -> figures/main/fig7_pk_vs_cross.{png,pdf}


---
The figures are in `figures/main/` as PNG (300 dpi) and PDF (vector, editable text).
`results/README.md` maps every figure to the file holding its numbers.

The per-layer versions of these metrics are in `figures/supplementary/`, grouped by
metric, and the significance tests between models are in `figures/statistics/`:

```bash
python run_pipeline.py figures --which supplementary
python run_pipeline.py figures --which statistics
```